In [57]:
import pandas as pd
import openpyxl
from datetime import date
import os.path

import gspread
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

def main():
    df_raw = get_excel_data(r'D:\Renstra\20260406 - KK Highlevel Real Mar 2026.xlsx', 'LR-SEMEN', "B, F, U, AJ", 2, 99, 2)
    df_Semen = get_rkap(df_raw)
    result = append_sheet('1taze3KNq8g1U76oSjp1yN_jMu6lzCZJiin0BQX5LzTs', 'Sheet1!A:D', 'https://www.googleapis.com/auth/spreadsheets', connect_to_sheet(), df_Semen)
    df_semen2 = get_rkap_normalize(df_raw)
    print(df_semen2)
    result = append_sheet('1taze3KNq8g1U76oSjp1yN_jMu6lzCZJiin0BQX5LzTs', 'Sheet2!A:D', 'https://www.googleapis.com/auth/spreadsheets', connect_to_sheet(), df_semen2)
    
                       
def get_excel_data(address, sheet_name=0, usecols=None, skiprows=0, nrows=None, header=None):
    df = pd.read_excel(
        address,
        sheet_name=sheet_name,
        usecols=usecols,
        skiprows=skiprows,
        nrows=nrows,
        dtype=str,
        thousands='.', 
        decimal=','
    )
    return df

def connect_to_sheet():
    SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists(r'C:\Users\mochamad.ilmawan\token.json'):
        creds = Credentials.from_authorized_user_file(r'C:\Users\mochamad.ilmawan\token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'D:\OneDrive\Documents\Direct Selling\MONITORING\Monitoring 2023\client_secret_738271294618-6g8e0hle4jpq8nau1d7b3q9jfsgp0ruk.apps.googleusercontent.com.json', SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open(r'C:\Users\mochamad.ilmawan\token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

def append_sheet(spreadSheetId, spread_range, scopes, creds, df):
    header = df.columns.tolist()
    data = df.values.tolist()
    
    # Combine them: [header] creates a list within a list to match the Sheets API format
    values_with_header = [header] + data
    try:
        values = data
        service = build('sheets', 'v4', credentials=creds)
        body = {
            'values': data
        }
        result = service.spreadsheets().values().append(
            spreadsheetId=spreadSheetId, range=spread_range, 
            valueInputOption="USER_ENTERED", body=body).execute()
        print(f"{(result.get('updates').get('updatedCells'))} cells appended.")
        return result

    except HttpError as error:
        print(f"An error occurred: {error}")
        return error
    
def get_rkap(df):
    # --- CLEANING PROCESS ---
    df.columns = ['Account','PREV','RKAP','REAL']
    start_idx = df[df['Account'] == "VAR COGS"].index[0]
    # 2. Find the index where "Beban Keuangan & Lainnya" ends
    end_idx = df[df['Account'] == "Beban Keuangan & Lainnya"].index[0]
    # 3. Drop the range (inclusive)
    # We use end_idx + 1 because Python slicing is exclusive of the last number
    df = df.drop(df.index[start_idx : end_idx + 1])
    # 4. Reset index to keep the data clean
    df = df.reset_index(drop=True)
    
    
    df = df.iloc[1:].reset_index(drop=True)
    df.dropna(subset=['Account'], inplace=True)
    df['Account'] = df['Account'].astype(str).str.rstrip()
    
    def clean_currency(value):
        return pd.to_numeric(value, errors='coerce')

    numeric_cols = ['PREV','RKAP', 'REAL']
    for col in numeric_cols:
        df[col] = df[col].apply(clean_currency)
        
    df_prev = df[['Account', 'PREV']].copy()
    df_prev.rename(columns={'PREV': 'Value'}, inplace=True)
    df_prev['Account'] = df_prev['Account'] + ' PREV'
    
    df_rkap = df[['Account', 'RKAP']].copy()
    df_rkap.rename(columns={'RKAP': 'Value'}, inplace=True)
    df_rkap['Account'] = df_rkap['Account'] + ' RKAP'

    # 2. Prepare df_real
    df_real = df[['Account', 'REAL']].copy()
    df_real.rename(columns={'REAL': 'Value'}, inplace=True)
    df_real['Account'] = df_real['Account']+ ' REAL'

    # 3. Combine them vertically (stacking rows)
    df = pd.concat([df_prev, df_rkap, df_real], axis=0).reset_index(drop=True)
    
    # 4. Add the date back (optional)
    df['date'] = '31/03/2026'
    df = df.fillna(0)
    
    # 4. Drop completely empty rows
    df = df.dropna(how='all')
    df = make_accounts_unique(df)
    df_pivoted = df.pivot_table(
        index=['date'], 
        columns='Account', 
        values='Value',
        aggfunc='first' # Use 'first' to keep the value as is
    ).reset_index()
   
    # 2. Optional: Clean up column names
    # Sometimes pivoting creates a 'Name' for the columns axis, we remove it
    df_pivoted.columns.name = None
    
    #print(df_pivoted)
    #print(df_pivoted.head())
    return df_pivoted.reset_index(drop=True)

def make_accounts_unique(df):
    # 1. Create a counter for each 'Account' within the same 'date' and 'Category'
    # This gives '0' to the first occurrence, '1' to the second, etc.
    df['occurrence'] = df.groupby(['date', 'Account']).cumcount()

    # 2. Append the number only if it is a duplicate (occurrence > 0)
    # This keeps "Gross Revenue" as is, but turns the next one into "Gross Revenue_2"
    df['Account'] = df.apply(
        lambda x: f"{x['Account']}_{x['occurrence'] + 1}" if x['occurrence'] > 0 else x['Account'], 
        axis=1
    )

    # 3. Drop the temporary counter column
    df = df.drop(columns=['occurrence'])
    
    return df

def get_rkap_normalize(df):
    # --- CLEANING PROCESS ---
    df.columns = ['Account','PREV','RKAP','REAL']
    start_idx = df[df['Account'] == "VAR COGS"].index[0]
    # 2. Find the index where "Beban Keuangan & Lainnya" ends
    end_idx = df[df['Account'] == "Beban Keuangan & Lainnya"].index[0]
    # 3. Drop the range (inclusive)
    # We use end_idx + 1 because Python slicing is exclusive of the last number
    df = df.drop(df.index[start_idx : end_idx + 1])
    # 4. Reset index to keep the data clean
    df = df.reset_index(drop=True)
    
    df = df.iloc[1:].reset_index(drop=True)
    df.dropna(subset=['Account'], inplace=True)
    df['Account'] = df['Account'].astype(str).str.rstrip()
    
    def clean_currency(value):
        return pd.to_numeric(value, errors='coerce')

    numeric_cols = ['PREV','RKAP', 'REAL']
    

    for col in numeric_cols:
        df[col] = df[col].apply(clean_currency)
    
    try:        
        cogs_list = ['Bahan Baku & Penolong', 'Bahan Bakar', 'Listrik', 'Pemeliharaan', 'Urusan Umum & Adm. Kantor',
                    'Kemasan', 'Perniagaan', 'Selisih Persediaan']
        
        
        for col in cogs_list:
            # 3. Create a new row as a Dictionary
            new_row = {
                'Account': col + 'per ton',
                'PREV': (df.loc[df['Account'] == col, 'PREV'].values[0])/(df.loc[df['Account'] == 'Total Sales Volume (million Ton)', 'PREV'].values[0])*1000,      
                'RKAP': (df.loc[df['Account'] == col, 'RKAP'].values[0])/(df.loc[df['Account'] == 'Total Sales Volume (million Ton)', 'RKAP'].values[0])*1000,
                'REAL': (df.loc[df['Account'] == col, 'REAL'].values[0])/(df.loc[df['Account'] == 'Total Sales Volume (million Ton)', 'REAL'].values[0])*1000,
            }
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        
    except IndexError:
        print("Warning: 'Biaya' or 'Volume' not found in the Account column.")
    df['date'] = '31/03/2026'
    df = make_accounts_unique(df)
    
    return df
    
def categorizing(df):
    '''
     df['Category'] = ""
    df.iloc[[0:3,5:6], df.columns.get_loc('Category')] = "revenue"
    df.iloc[4, df.columns.get_loc('Category')] = "ongkos angkut"
    df.iloc[5:17, df.columns.get_loc('Category')] = "COGS"
    df.iloc[17, df.columns.get_loc('Category')] = "laba kotor"
    df.iloc[20:28, df.columns.get_loc('Category')] = "beban umum"
    df.iloc[30:39, df.columns.get_loc('Category')] = "beban pemasaran"
    df.iloc[43:47, df.columns.get_loc('Category')] = "other"
    '''
   
    
    
if __name__ == '__main__':
    main()

                             Account          PREV          RKAP  \
0     Gross Revenue (Cement+Clinker)  1.977055e+06  2.022134e+06   
1               Gross Revenue Mortar  4.554017e+03  9.994814e+03   
2                      Gross Revenue  1.981609e+06  2.032129e+06   
3                      Ongkos Angkut -1.346450e+05 -1.336201e+05   
4                            Revenue  1.846964e+06  1.898509e+06   
..                               ...           ...           ...   
72               Pemeliharaanper ton -5.860128e+01 -6.014704e+01   
73  Urusan Umum & Adm. Kantorper ton -1.468341e+01 -1.648361e+01   
74                    Kemasanper ton -2.570522e+01 -3.363564e+01   
75                 Perniagaanper ton -6.036622e+01 -5.640468e+01   
76         Selisih Persediaanper ton  9.763237e+01  1.637780e+01   

            REAL        date  
0   1.974876e+06  31/03/2026  
1   4.015977e+03  31/03/2026  
2   1.978892e+06  31/03/2026  
3  -1.442129e+05  31/03/2026  
4   1.834679e+06  31/03/2026